In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
class Chunker:
  def __init__(self,chunk_size=800,overlap=100):
    self.chunk_size = chunk_size
    self.overlap = overlap

  def split(self,text):
    chunks = []
    start = 0
    while start < len(text):
      chunk = text[start:start+ self.chunk_size]
      chunks.append(chunk)
      start += self.chunk_size - self.overlap
    return chunks

In [ ]:
sample = "A" * 2000
chunker = Chunker(chunk_size=800, overlap=100)
chunks = chunker.split(sample)
print("Number of pieces:", len(chunks))
print("Length of the first piece:", len(chunks[0]))

Number of pieces: 3
Length of the first piece: 800


In [ ]:
class Embedder:
  def __init__(self, model_name = "all-MiniLM-L6-v2"):
    self.model = SentenceTransformer(model_name)

  def encode(self, texts):
    return self.model.encode(texts)


In [ ]:
embedder = Embedder()

sentences = [
    "the fuel consumption of the car",
    "the gasoline usage of the automobile",
    "the sleeping pattern of cats"
]

vectors = embedder.encode(sentences)
print("Shape:", vectors.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Shape: (3, 384)


In [ ]:
import numpy as np
def similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("vehicle-automobile (similar):", similarity(vectors[0], vectors[1]))
print("vehicle-cat (unrelated):  ",similarity(vectors[0], vectors[2]))

vehicle-automobile (similar): 0.7688451
vehicle-cat (unrelated):   0.12943031


In [ ]:
!pip install faiss-cpu -q

In [ ]:
import faiss
import numpy as np

In [ ]:
class VectorStore:
  def __init__(self,dim=384):
    self.index = faiss.IndexFlatIP(dim) #kosinüs için index IP= inner product normalize + dot product
    self.texts= []

  def add(self, vectors, texts):
    faiss.normalize_L2(vectors)
    self.index.add(vectors)
    self.texts.extend(texts)

  def search(self,query_vector, k=5):
    faiss.normalize_L2(query_vector)
    scores, idx = self.index.search(query_vector, k)
    return [self.texts[i] for i in idx[0]]


In [ ]:
sentences =  [
    "the fuel consumption of the car",
    "the gasoline usage of the automobile",
    "the sleeping pattern of cats",
    "how to train a neural network",
]

vectors = embedder.encode(sentences)

store = VectorStore(dim=384)
store.add(vectors, sentences)

question = embedder.encode(["car fuel economy"])
results = store.search(question, k=2)
print(results)

['the fuel consumption of the car', 'the gasoline usage of the automobile']


In [ ]:
from sentence_transformers import CrossEncoder

In [ ]:
class Reranker:
  def __init__(self, model_name="cross-encoder/ms-marco-MiniLM-L-6-v2"):
    self.model = CrossEncoder(model_name)

  def rerank(self, query, candidates, k=5):
    pairs = [(query, c) for c in candidates]
    scores = self.model.predict(pairs)
    top = np.argsort(scores)[::-1][:k] #ilk index ters çevirir, ikincisi ilk k
    return [candidates[i] for i in top]

In [ ]:
candidates = store.search(embedder.encode(["car fuel economy"]), k=4)
print("Candidate (embedding rank):", candidates)

reranker = Reranker()
the_best = reranker.rerank("car fuel economy",candidates,k=2)
print("After rerank (best of two):",the_best)

Candidate (embedding rank): ['the fuel consumption of the car', 'the gasoline usage of the automobile', 'how to train a neural network', 'the sleeping pattern of cats']


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

After rerank (best of two): ['the fuel consumption of the car', 'the gasoline usage of the automobile']


In [ ]:
from transformers import pipeline

In [ ]:
class Generator:
    def __init__(self,model_name="Qwen/Qwen2.5-1.5B-Instruct" ):
      #eski model model_name="Qwen/Qwen2.5-0.5B-Instruct" , daha güçlü cevap almak için değiştirildi
        self.llm = pipeline("text-generation", model=model_name)

    def generate(self, query, chunks):
        context = "\n\n".join(chunks) #araya 2 yeni satır ekler

        prompt = f"""Answer the question based ONLY on the context below.
If the answer is not in the context, say "I don't know". Don't make things up.
Answer in one or two complete sentences based on the context.
Context:
{context}

Question: {query}
Answer:"""

        messages = [{"role": "user", "content": prompt}]
        output = self.llm(messages, max_new_tokens=256)
        return output[0]["generated_text"][-1]["content"]

In [ ]:
#Retrival+rerank+generation
question = "What is the fuel consumption of the car?"
candidates = store.search(embedder.encode([question]), k=4)
the_best = reranker.rerank(question,candidates, k=2)

generator = Generator()
answer = generator.generate(question,the_best)
print("Answer:",answer)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Answer: The fuel consumption of the car refers to how much fuel it uses per unit time, typically measured in liters per 100 kilometers (L/100 km) or miles per gallon (mpg).


In [ ]:
class RAGPipeline:
  def __init__(self):
    self.chunker = Chunker()
    self.embedder = Embedder()
    self.store = VectorStore()
    self.reranker = Reranker()
    self.generator = Generator()

  def index(self,documents):
    all_chunks = []
    for doc in documents:
      chunks = self.chunker.split(doc)
      all_chunks.extend(chunks)
    vectors= self.embedder.encode(all_chunks)
    self.store.add(vectors,all_chunks)

  def query(self,question):
    q = self.embedder.encode([question])
    candidates = self.store.search(q,k=4)
    the_best = self.reranker.rerank(question, candidates, k=2)
    return self.generator.generate(question, the_best)

  def retrieve(self,question, k=5):
    q = self.embedder.encode([question])
    candidates = self.store.search(q, k=20)
    return self.reranker.rerank(question, candidates, k=k)




In [ ]:
documents = [
    "The Toyota Corolla has a fuel consumption of 6.5 liters per 100 km. It is a compact sedan.",
    "Cats sleep around 15 hours a day. They are crepuscular animals.",
    "A neural network is trained using backpropagation and gradient descent.",
]

rag = RAGPipeline()
rag.index(documents)
print(rag.query("What is the fuel consumption of the Corolla?"))
print(rag.query("How many hours do cats sleep?"))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The fuel consumption of the Toyota Corolla is 6.5 liters per 100 kilometers.
Cats sleep around 15 hours a day.


In [ ]:
!pip install pypdf -q
from pypdf import PdfReader

In [ ]:
def pdf_to_text(path):
  reader = PdfReader(path)
  text = ""
  for page in reader.pages:
    text+= page.extract_text()

  return text


In [ ]:
paper1 = pdf_to_text("paper1.pdf")
paper2 = pdf_to_text("paper2.pdf")
print(len(paper1),len(paper2))

rag = RAGPipeline()
rag.index([paper1,paper2])

print(rag.query("How many benchmarks does Strategist beat the baseline on?"))
print(rag.query("What approach does the Zebra puzzle paper propose?"))


79805 93930


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Based on the information provided in the context, Strategist beats the baseline on 24 out of the 30 benchmarks.
The Zebra puzzle paper proposes a method called NeuralExactCover for solving complex combinatorial reasoning tasks, particularly ZebraPuzzles.


In [ ]:
eval_set = [
    {"q": "How many benchmarks does Strategist beat the baseline on?", "expect": "24"},
    {"q": "What accuracy does Strategist achieve?",                      "expect": "74.7"},
    {"q": "How much does Strategist reduce inference cost?",             "expect": "39"},
    {"q": "What kind of puzzles does the first paper solve?",            "expect": "Zebra"},
    {"q": "Is the first paper's approach neuro-symbolic?",               "expect": "neuro-symbolic"},
]

In [ ]:
hits = 0
for item in eval_set:
  chunks = rag.retrieve(item["q"], k=5)
  joint = " ".join(chunks) #parçaları tek metne birleştir
  if item["expect"].lower() in joint.lower():
    hits += 1
    print("okey",item["q"])
  else:
    print("not exist",item["q"])

print(f"\nHit@5: {hits}/{len(eval_set)}")

okey How many benchmarks does Strategist beat the baseline on?
okey What accuracy does Strategist achieve?
okey How much does Strategist reduce inference cost?
okey What kind of puzzles does the first paper solve?
okey Is the first paper's approach neuro-symbolic?

Hit@5: 5/5
